# <font color='black'>Регрессионный анализ панельных данных и каузальность, 2026 </font>
## <font color='black'>Практическое занятие 6. Модели со смешанными эффектами: адаптация к панельным данным </font>


Для анализа мы будем использовать данные исследования National Youth Survey. Мы проследим, как склонность к девиантному поведению меняется среди подростков во временной перспективе, и какие факторы обуславливают эти изменения. В рамках исследования опрашивались подростки возраста 14 - 18 лет (лонгитюдные данные). Ниже представлено краткое описание переменных:

"The dependent variable attit is a 9-item scale assessing attitudes favorable to deviant behavior (property damage, drug and alcohol use, stealing, etc.).

The level-1 independent variables include: expo measuring
exposure to deviant peers (students were asked how many of their friends engaged in the deviant behaviors) and age (age in years). Level 2 include person-level variables: female, minority, and income"

Подгрузим необходимые библиотеки и откроем массив "nys.dta".

In [ ]:
import pandas as pd
import numpy as np
import math
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
%matplotlib inline
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLMParams

In [ ]:
mixed = pd.read_stata('nys.dta')
mixed.head(10)

,id,attit14,expo14,attit15,expo15,attit16,expo16,attit17,expo17,attit18,expo18,female,minority,income
0,5,NaN,NaN,NaN,NaN,0.58,0.64,0.64,0.58,0.89,1.06,1,0,1
1,7,0.58,0.94,0.29,0.80,0.11,0.44,0.44,0.58,0.51,0.58,1,0,3
2,11,0.69,0.51,0.69,0.58,0.80,0.85,NaN,NaN,NaN,NaN,0,0,3
3,50,0.94,1.02,0.94,1.24,1.10,1.20,NaN,NaN,NaN,NaN,0,0,5
4,54,0.44,1.02,0.51,0.85,0.69,1.30,0.58,1.02,0.58,0.85,1,0,1
5,56,NaN,NaN,NaN,NaN,0.37,0.10,0.11,0.51,NaN,NaN,1,1,1
6,60,NaN,NaN,0.69,0.29,0.44,0.29,0.64,0.20,0.58,0.20,1,1,3
7,84,0.44,0.85,0.69,1.06,0.89,0.69,0.51,0.85,0.37,0.36,0,1,3
8,87,0.85,0.58,NaN,NaN,0.75,0.92,NaN,NaN,NaN,NaN,0,1,1
9,89,0.29,0.00,NaN,NaN,0.00,0.10,0.29,0.29,0.29,0.94,0,1,2


Исходный массив представлен в так называемом широком формате: в строке для каждого id представлены значения переменных attit и expo -- ответы подростков, зафиксированные в разном возрасте. Для последующего анализа преобразуем формат массива в длинный: каждая строка будет соответствовать теперь уже не каждому отдельному индивиду (подростку, принимающему участие в опросе), а индивиду определенного возраста (к примеру, ответы id 84 мы последовательно проследим в 14, 15, 16, 17 и 18 лет, и разместим эти ответы не в одной строке, а в пяти строках для id 84).

In [ ]:
long = pd.wide_to_long(mixed, ['attit', 'expo'], i='id',
                j ='age', sep='').reset_index()

long.head()

,id,age,female,income,minority,attit,expo
0,5,14,1,1,0,NaN,NaN
1,7,14,1,3,0,0.58,0.94
2,11,14,0,3,0,0.69,0.51
3,50,14,0,5,0,0.94,1.02
4,54,14,1,1,0,0.44,1.02


In [ ]:
long[long['id'] == 84]

,id,age,female,income,minority,attit,expo
7,84,14,0,3,1,0.44,0.85
248,84,15,0,3,1,0.69,1.06
489,84,16,0,3,1,0.89,0.69
730,84,17,0,3,1,0.51,0.85
971,84,18,0,3,1,0.37,0.36


In [ ]:
long = long.dropna()

long.head()

,id,age,female,income,minority,attit,expo
1,7,14,1,3,0,0.58,0.94
2,11,14,0,3,0,0.69,0.51
3,50,14,0,5,0,0.94,1.02
4,54,14,1,1,0,0.44,1.02
7,84,14,0,3,1,0.44,0.85


In [ ]:
figure = px.line(data_frame=long, x="age", y="attit", animation_frame="id")
figure.update_yaxes(range=[0, 1.5])

figure

1) Оцените ANOVA-модель, в которой зависимой переменной является attit. Рассчитайте значение ICC и проинтерпретируйте полученное значение

2) Перекодируйте переменную age таким образом, чтобы минимальному возрасту, доступному в массиве, соответствовало значение 0

3) Смоделируйте зависимость attit от возраста. Протестируйте в модели со смешанными эффектами как линейный эффект возраста, так и квадратичный. При необходимости включите случайные эффекты. Протестируйте альтернативные спецификации моделей

4) Включите на следующем шаге объясняющие переменные в модель expo, income, female, minority в качестве фиксированных эффектов. Проинтерпретируйте полученные результаты